# NI USB-6002 → Python Live Data Logger (simple)

This notebook does two things:

1. **Live view** your voltage signal(s) while recording  
2. **Export to CSV** for Excel / Google Sheets

> If you already have the **NI-DAQmx driver** installed and the device shows up in **NI MAX**, you’re 90% of the way there.

---

## 0) Plotting mode (important for live view)

- **Best option (recommended):** `widget` backend (smooth live updates)
- **Fallback:** `inline` backend (still works, but live updates can look “choppy”)

If you see an error about `ipympl` / `jupyter-matplotlib`, switch to `inline` below.

---

In [ ]:
# Recommended for live updates in Jupyter:
%matplotlib widget

# If you get an ipympl/jupyter-matplotlib error, comment the line above
# and uncomment the line below:
# %matplotlib inline


## 1) Imports

In [ ]:
import time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nidaqmx
from nidaqmx.constants import AcquisitionType, TerminalConfiguration
from nidaqmx.stream_readers import AnalogMultiChannelReader
from nidaqmx.system import System


## 2) Detect NI DAQ devices

If you see **no devices**, open **NI MAX** and confirm the USB-6002 shows up there first.


In [ ]:
system = System.local()
print("NI-DAQmx driver version:", system.driver_version)

if len(system.devices) == 0:
    raise RuntimeError("No NI DAQ devices found. Check NI MAX and your USB connection.")
else:
    for dev in system.devices:
        print("Device:", dev.name, "| Product:", dev.product_type)


## 3) Logger settings (edit these)

**Student edit zone** ✅

Notes:
- USB-6002 analog inputs are typically **±10 V**.
- Choose **RSE** vs **DIFF** to match how you wired the sensor. If you’re unsure, start with **RSE**.


In [ ]:
# -------------------------
# Hardware (edit this)
# -------------------------
DEVICE = "Dev1"                 # <-- from the detection cell above
CHANNELS = ["ai0"]              # e.g., ["ai0"] or ["ai0","ai1","ai2"]
PHYS_CHANS = [f"{DEVICE}/{ch}" for ch in CHANNELS]

# -------------------------
# Sampling (edit this)
# -------------------------
SAMPLE_RATE_HZ = 1000.0         # Hz
DURATION_S = 10.0               # seconds
CHUNK_SAMPLES = 200             # samples read per loop (tradeoff: smoother vs CPU)

# -------------------------
# Input range (edit if needed)
# -------------------------
MIN_V = -10.0
MAX_V = 10.0

# -------------------------
# Wiring mode (edit this)
# -------------------------
TERM_CFG = TerminalConfiguration.RSE
# TERM_CFG = TerminalConfiguration.DIFFERENTIAL

# -------------------------
# Live view (edit if desired)
# -------------------------
LIVE_WINDOW_S = 5.0             # show only the most recent N seconds
PLOT_EVERY_CHUNK = True         # if False, records without live plot updates


## 4) Live record + log (main cell)

Run this cell to:
- start acquisition
- update a live plot while recording
- store all samples in memory for CSV export

Stop early: **Kernel → Interrupt** (or press the stop button).


In [ ]:
# -------------------------
# Live acquisition + logging
# -------------------------
n_ch = len(PHYS_CHANS)
fs = float(SAMPLE_RATE_HZ)

num_total_samples = int(round(fs * float(DURATION_S)))
num_chunks = int(np.ceil(num_total_samples / CHUNK_SAMPLES))

# Pre-allocate arrays for the full run (fast + simple)
t = np.empty(num_total_samples, dtype=np.float64)
data = np.empty((n_ch, num_total_samples), dtype=np.float64)

# Live plot buffers (just for display)
win_n = max(10, int(round(LIVE_WINDOW_S * fs)))
tbuf = np.empty(win_n, dtype=np.float64)
vbuf = np.empty((n_ch, win_n), dtype=np.float64)
tbuf[:] = np.nan
vbuf[:] = np.nan

# Plot setup
fig, ax = plt.subplots()
lines = []
for i, ch in enumerate(CHANNELS):
    (ln,) = ax.plot([], [], label=ch)
    lines.append(ln)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (V)")
ax.set_title("Live View (NI USB-6002)")
ax.grid(True)
ax.legend(loc="upper right")
plt.show()

start_wall = datetime.now()
t0 = time.perf_counter()

# Helper: update plot using the last window of data
def _update_plot(t_window, v_window):
    if not PLOT_EVERY_CHUNK:
        return
    # Update each line
    for i in range(n_ch):
        lines[i].set_data(t_window, v_window[i, :])

    # X limits
    if np.isfinite(t_window).any():
        xmin = np.nanmin(t_window)
        xmax = np.nanmax(t_window)
        if xmin == xmax:
            xmax = xmin + 1e-6
        ax.set_xlim(xmin, xmax)

        # Y limits (auto, with a little padding)
        vmin = np.nanmin(v_window)
        vmax = np.nanmax(v_window)
        if np.isfinite(vmin) and np.isfinite(vmax):
            if vmin == vmax:
                vmin -= 0.5
                vmax += 0.5
            pad = 0.05 * (vmax - vmin)
            ax.set_ylim(vmin - pad, vmax + pad)

    fig.canvas.draw_idle()
    fig.canvas.flush_events()

# Acquisition
with nidaqmx.Task() as task:
    # Add channels
    for ch in PHYS_CHANS:
        task.ai_channels.add_ai_voltage_chan(
            ch,
            min_val=MIN_V,
            max_val=MAX_V,
            terminal_config=TERM_CFG,
        )

    # Timing
    task.timing.cfg_samp_clk_timing(
        rate=fs,
        sample_mode=AcquisitionType.CONTINUOUS,
        samps_per_chan=CHUNK_SAMPLES,
    )

    reader = AnalogMultiChannelReader(task.in_stream)

    task.start()
    idx = 0

    try:
        for _ in range(num_chunks):
            n = min(CHUNK_SAMPLES, num_total_samples - idx)
            buf = np.empty((n_ch, n), dtype=np.float64)

            # Blocks until n samples are available
            reader.read_many_sample(
                data=buf,
                number_of_samples_per_channel=n,
                timeout=10.0,
            )

            # Store full run
            data[:, idx:idx+n] = buf
            t[idx:idx+n] = np.arange(idx, idx+n) / fs

            # Update live window buffers (fast, no Python loops)
            if win_n <= n:
                # If the chunk is bigger than the window, only keep the tail
                tbuf[:] = t[idx+n-win_n:idx+n]
                vbuf[:, :] = data[:, idx+n-win_n:idx+n]
            else:
                # Shift left and append
                shift = n
                tbuf[:-shift] = tbuf[shift:]
                vbuf[:, :-shift] = vbuf[:, shift:]
                tbuf[-shift:] = t[idx:idx+n]
                vbuf[:, -shift:] = buf

            _update_plot(tbuf, vbuf)

            idx += n

    except KeyboardInterrupt:
        print("Stopped early by user.")
        # Trim arrays to what we collected
        t = t[:idx].copy()
        data = data[:, :idx].copy()

    finally:
        task.stop()

elapsed = time.perf_counter() - t0
print(f"Acquired {data.shape[1]} samples on {n_ch} channel(s) @ {fs:.1f} Hz in {elapsed:.2f} s")


## 5) Export to CSV ✅

This creates a tidy table:

- `time_s`
- one column per channel (e.g., `ai0_V`, `ai1_V`, ...)



In [ ]:
# Build DataFrame
out = {"time_s": t}
for i, ch in enumerate(CHANNELS):
    out[f"{ch}_V"] = data[i, :]

df = pd.DataFrame(out)

fname = f"usb6002_{DEVICE}_" + "_".join(CHANNELS) + f"_{start_wall.strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(fname, index=False)
print("Wrote:", fname)

df.head()


## 6) Quick post-run plot (optional)

Use this if you want a clean plot after the recording (not “live”).


In [ ]:
plt.figure()
for ch in CHANNELS:
    plt.plot(df["time_s"], df[f"{ch}_V"], label=ch)
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("Recorded Data")
plt.grid(True)
plt.legend()
plt.show()


## 7) Troubleshooting (common student issues)

### A) “No NI DAQ devices found”
- Confirm the USB-6002 shows in **NI MAX**
- Try a different USB port/cable
- Reboot (yes, really)

### B) “Device is reserved / resource is in use”
- Close **NI MAX** test panels (they can “lock” the device)
- Restart the Jupyter kernel (Kernel → Restart)

### C) Wiring mode confusion (RSE vs DIFF)
- If your sensor has a **single-ended output referenced to DAQ GND**, start with **RSE**  
- If you wired **+ and –** to a pair and want the DAQ to measure the difference, use **DIFFERENTIAL**

### D) Flat line / nonsense data
- Check you’re on the correct physical channel (`ai0`, `ai1`, ...)
- Check your signal is within the **MIN_V/MAX_V** range
- Confirm your sensor ground is tied correctly

### E) Live plot not updating
- If `%matplotlib widget` errors, switch to `%matplotlib inline`
- Reduce `SAMPLE_RATE_HZ` or increase `CHUNK_SAMPLES` if your computer is struggling

---